# Legislative Content Extraction (ADK) - Evaluation Smoke Test

Fetch the dataset from Langfuse and run the agent on the first item to verify the pipeline works end-to-end.

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# Ensure cwd is the repo root so relative paths work
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
# Walk up until we find pyproject.toml
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

load_dotenv(verbose=True)
print(f"Working directory: {Path.cwd()}")

Working directory: /home/coder/eval-agents


## 1. Fetch dataset from Langfuse

In [2]:
from aieng.agent_evals.async_client_manager import AsyncClientManager

client_manager = AsyncClientManager.get_instance()
langfuse_client = client_manager.langfuse_client

DATASET_NAME = "legislative-docs-ID"
FILES_DIR = Path("implementations/legislative_content_extraction/files").resolve()
MAX_CONCURRENCY = 3

dataset = langfuse_client.get_dataset(DATASET_NAME)
print(f"Dataset name: {dataset.name}")
print(f"Items:   {len(dataset.items)}")
print(f"Files dir:     {FILES_DIR}")
print(f"Concurrency:   {MAX_CONCURRENCY}")

Dataset name: legislative-docs-ID
Items:   6
Files dir:     /home/coder/eval-agents/implementations/legislative_content_extraction/files
Concurrency:   3


## 2. Inspect the first item

In [3]:
first_item = dataset.items[0]

print("Input:")
print(json.dumps(first_item.input, indent=2))
print("\nExpected output:")
print(json.dumps(first_item.expected_output, indent=2))

Input:
{
  "prompt": "Extract legislative metadata from this bill.",
  "record_id": "ID_S1331",
  "pdf_file_name": "ID_S1331.pdf",
  "html_page_link": "https://legislature.idaho.gov/sessioninfo/2026/legislation/S1331/"
}

Expected output:
{
  "title": "An Act relating to appropriations; reducing the appropriation for fiscal year 2026; reducing the number of authorized full-time equivalent positions for fiscal year 2026; transferring moneys from the    \n  Public School Income Fund to the General Fund for fiscal year 2026; and declaring an emergency.",
  "summary": "Senate Bill 1331 (2026) is an emergency appropriations reduction bill that cuts $192,656,100 across virtually every branch and agency of Idaho state government for fiscal year 2026 (July 1, 2025 - June 30, 2026).\n  Sponsored by the Senate Finance Committee, the bill reduces previously enacted FY2026 appropriations from the Laws of 2025 across 31 major budget areas including public schools ($22.4M), universities and colleges

## 3. Run evaluation on the dataset

In [4]:
from aieng.agent_evals.legislative_content_extraction.evaluation.offline import evaluate

await evaluate(
    dataset_name=DATASET_NAME,
    files_dir=FILES_DIR,
    max_concurrency=MAX_CONCURRENCY,
)

2026-03-24 21:59:29,645 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-03-24 21:59:29,689 INFO aieng.agent_evals.langfuse: Langfuse tracing initialized successfully (endpoint: https://us.cloud.langfuse.com/api/public/otel)
2026-03-24 21:59:29,691 WARNING google_genai._api_client: Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-03-24 21:59:30,082 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metadata from: /home/coder/eval-agents/implementations/legislative_content_extraction/files/ID_S1331/ID_S1331.pdf
2026-03-24 21:59:30,089 WARNING google_genai._api_client: Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-03-24 21:59:30,212 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-24 21:59:30,218 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metadata